<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab2_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M03 Lab 2 — Function Calling & Tool Use

**DADS 5250: Generative AI in Practice** | Prof. Dehghani

Difficulty: ⭐⭐ | Time: ~10 min

## Learning Objectives

1. Define function schemas for tool/function calling
2. Parse tool call responses programmatically
3. Complete the tool call loop (call → execute → return)
4. Build a real-world structured extractor

In [ ]:
!pip install -q openai tiktoken pydantic
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
import json
from pydantic import BaseModel
from openai import OpenAI
from dads5250 import setup_openai, show_response, show_expected, show_info, quiz

client = setup_openai()

---
## 1. Function Calling Basics

Function calling lets you define **tools** the model can invoke. Instead of text, the model returns a structured function call with arguments.

**Flow:** Define schema → Model decides to call → You execute → Feed result back

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"},
                    "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}
                },
                "required": ["city"]
            }
        }
    }
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "What's the weather like in Boston today?"}],
    tools=tools,
    tool_choice="auto",
)

tool_call = response.choices[0].message.tool_calls[0]
print(f"Function: {tool_call.function.name}")
print(f"Arguments: {tool_call.function.arguments}")

args = json.loads(tool_call.function.arguments)
print(f"\nParsed -> city: {args['city']}, units: {args.get('units', 'not specified')}")

---
## 2. Completing the Tool Call Loop

After the model requests a tool call, execute the function and send the result back.

In [ ]:
def get_weather(city: str, units: str = "fahrenheit") -> dict:
    """Simulated weather function."""
    return {"city": city, "temp": 42, "units": units, "condition": "Partly cloudy"}

result = get_weather(**args)
print(f"Function result: {result}")

# Send result back to the model
follow_up = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "What's the weather like in Boston today?"},
        response.choices[0].message,
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result),
        }
    ],
    tools=tools,
)

show_response(follow_up)

---
## 3. Real-World Example: Resume Parser

Combine JSON mode + Pydantic to build a structured resume extractor.

In [ ]:
class Experience(BaseModel):
    company: str
    title: str
    duration: str

class ResumeData(BaseModel):
    name: str
    email: str
    skills: list[str]
    experience: list[Experience]
    education: str

resume_text = """
Jane Smith - jane.smith@email.com
Senior Data Scientist with 6 years of experience in ML and NLP.

EXPERIENCE
- DataCorp Inc. | Senior Data Scientist | 2021-Present
  Led NLP team, built production recommendation engine
- StartupAI | ML Engineer | 2019-2021
  Developed computer vision pipeline for quality inspection

SKILLS: Python, PyTorch, TensorFlow, SQL, NLP, Computer Vision, MLOps

EDUCATION: M.S. Computer Science, MIT (2019)
"""

r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Extract structured resume data. Return JSON with keys: name, email, skills (list), experience (list of {company, title, duration}), education."},
        {"role": "user", "content": resume_text}
    ],
    response_format={"type": "json_object"},
)

resume = ResumeData(**json.loads(r.choices[0].message.content))
print(f"Name: {resume.name}")
print(f"Email: {resume.email}")
print(f"Skills: {', '.join(resume.skills)}")
for exp in resume.experience:
    print(f"  - {exp.title} at {exp.company} ({exp.duration})")
print(f"Education: {resume.education}")

---
## 4. Knowledge Check

In [ ]:
quiz([
    {
        "q": "When should you use function calling over JSON mode?",
        "options": [
            "When you need the cheapest option",
            "When you want the model to decide which action to take from a set of available tools",
            "When you only need simple text responses",
            "When working with Gemini models"
        ],
        "answer": 1,
        "explanation": "Function calling is ideal when the model needs to choose between multiple tools and provide structured arguments — the foundation of AI agents."
    }
])

---
## 5. Hands-on: Define a Function Schema

Define a tool for `search_products` with: query (string), category (enum: electronics, clothing, books), max_price (number, optional).

In [ ]:
# Replace the _____ parts

search_tool = {
    "type": "function",
    "function": {
        "name": "_____",                    # YOUR CODE HERE
        "description": "_____",             # YOUR CODE HERE
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "_____",        # YOUR CODE HERE
                    "description": "Search query string"
                },
                "category": {
                    "type": "string",
                    "enum": ["_____", "_____", "_____"],    # YOUR CODE HERE
                    "description": "Product category"
                },
                "max_price": {
                    "type": "_____",        # YOUR CODE HERE
                    "description": "Maximum price filter"
                }
            },
            "required": ["query", "category"]
        }
    }
}

response_ex1 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Find me wireless headphones under $100"}],
    tools=[search_tool],
    tool_choice="auto",
)

tc = response_ex1.choices[0].message.tool_calls[0]
print(f"Function: {tc.function.name}")
print(f"Arguments: {tc.function.arguments}")

In [ ]:
# --- Test ---
assert tc.function.name == "search_products", f"Expected 'search_products', got '{tc.function.name}'"
args_ex1 = json.loads(tc.function.arguments)
assert "query" in args_ex1, "Missing 'query' in arguments"
show_expected('Function: search_products\nArguments: {"query": "wireless headphones", "category": "electronics", "max_price": 100}')

---
## Summary

- **Function calling**: Define tool schemas → model returns structured calls → you execute
- **Tool call loop**: user message → assistant tool_call → tool result → assistant response
- **Real-world**: Combine JSON mode + Pydantic for robust extraction pipelines

---

## Assignment M03

Build a **structured data extractor** for a domain of your choice (menu, job posting, product review, etc.).

**Requirements:**
1. Define a Pydantic model for your domain
2. Use JSON mode to extract data from at least 3 sample inputs
3. Validate outputs with Pydantic

**Submit:** Code + 3 sample inputs/outputs + 1-paragraph observation on edge cases